## **Camada Silver - silver.tb_info_filmes (origem: tb_movies_info)**

In [0]:
# Importa as funções do PySpark que serão utilizadas nas transformações
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import IntegerType

spark = SparkSession.builder.getOrCreate()

# Cria o database/schema "silver"
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

# Lê a tabela de filmes da camada Bronze
df_filmes = spark.table("bronze.tb_movies_info")

# Exibe algumas linhas para conferirmos os dados de origem
display(df_filmes.limit(20))

id,tconst,title,original_title,original_language,release_date,runtime,status,overview,tagline,ingestion_datetime
293660,tt1431045,Deadpool,Deadpool,en,2016-02-09,108,Released,"The origin story of former Special Forces operative turned mercenary Wade Wilson, who, after being subjected to a rogue experiment that leaves him with accelerated healing powers, adopts the alter ego Deadpool. Armed with his new abilities and a dark, twisted sense of humor, Deadpool hunts down the man who nearly destroyed his life.",Witness the beginning of a happy ending.,2026-10-07T07:55:29.373Z
299536,tt4154756,AVENGERS: INFINITY WAR,Avengers: Infinity War,en,04-25-2018,149,Released,"As the Avengers and their allies have continued to protect the world from threats too large for any one hero to handle, a new danger has emerged from the cosmic shadows: Thanos. A despot of intergalactic infamy, his goal is to collect all six Infinity Stones, artifacts of unimaginable power, and use them to inflict his twisted will on all of reality. Everything the Avengers have fought for has led up to this moment - the fate of Earth and existence itself has never been more uncertain.",An entire universe. Once and for all.,2026-10-07T07:55:29.373Z
299534,tt4154796,Avengers: Endgame,Avengers: Endgame,en,2019-04-24,181,released,"After the devastating events of Avengers: Infinity War, the universe is in ruins due to the efforts of the Mad Titan, Thanos. With the help of remaining allies, the Avengers must assemble once more in order to undo Thanos' actions and restore order to the universe once and for all, no matter what consequences may be in store.",Avenge the fallen.,2026-10-07T07:55:29.373Z
475557,tt7286456,Joker,Joker,en,2019-10-01,122,Released,"During the 1980s, a failed stand-up comedian is driven insane and turns to a life of crime and chaos in Gotham City while becoming an infamous psychopathic crime figure.",Put on a happy face.,2026-10-07T07:55:29.373Z
271110,tt3498820,Captain America: Civil War,Captain America: Civil War,en,2016-04-27,147,Released,"Following the events of Age of Ultron, the collective governments of the world pass an act designed to regulate all superhuman activity. This polarizes opinion amongst the Avengers, causing two factions to side with Iron Man or Captain America, which causes an epic battle between former allies.",United we stand. Divided we fall.,2026-10-07T07:55:29.373Z
284054,tt1825683,Black Panther,Black Panther,en,2018-02-13,135,Released,"King T'Challa returns home to the reclusive, technologically advanced African nation of Wakanda to serve as his country's new leader. However, T'Challa soon finds that he is challenged for the throne by factions within his own country as well as without. Using powers reserved to Wakandan kings, T'Challa assumes the Black Panther mantle to join with ex-girlfriend Nakia, the queen-mother, his princess-kid sister, members of the Dora Milaje (the Wakandan 'special forces') and an American secret agent, to prevent Wakanda from being dragged into a world war.",null,2026-10-07T07:55:29.373Z
284052,tt1211837,Doctor Strange,Doctor Strange,en,2016-10-25,115,Released,"After his career is destroyed, a brilliant but arrogant surgeon gets a new lease on life when a sorcerer takes him under her wing and trains him to defend the world against evil.",The impossibilities are endless.,2026-10-07T07:55:29.373Z
315635,tt2250912,Spider-Man: Homecoming,Spider-Man: Homecoming,en,2017-07-05,133,RELEASED,"Following the events of Captain America: Civil War, Peter Parker, with the help of his mentor Tony Stark, tries to balance his life as an ordinary high school student in Queens, New York City, with fighting crime as his superhero alter ego Spider-Man as a new threat, the Vulture, emerges.",Homework can wait. The city can't.,2026-10-07T07:55:29.373Z
283995,tt3896198,Guardians of the Galaxy Vol. 2,Guardians of the Galaxy Vol. 2,en,2017-04-19,137,Released,The Guardians must fight to keep their newfound family together as t

In [0]:
# Normaliza a coluna 'status' original: remove ruídos, hífens e padroniza a caixa
df_filmes = df_filmes.withColumn("status_normalizado", F.trim(F.lower(F.regexp_replace(F.col("status"), "-", " "))))

# Tradução dos termos e padronização dos não mapeáveis como "Não Informado"
df_filmes = df_filmes.withColumn(
    "status_filme",
    F.when(F.col("status_normalizado") == "released", "Lançado")
     .when(F.col("status_normalizado") == "post production", "Pós-Produção")
     .when(F.col("status_normalizado") == "in production", "Em Produção")
     .when(F.col("status_normalizado") == "planned", "Planejado")
     .when(F.col("status_normalizado") == "rumored", "Rumores")
     .when(F.col("status_normalizado") == "canceled", "Cancelado")
     .otherwise("Não Informado")
)

# Cria uma janela agrupando os registros pelo ID do filme e ordenando pela data de ingestão da mais recente para a mais antiga
janela_filmes = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())

# Ordena cada filme pela ingestão mais recent e mantém apenas a versão mais recente de cada filme
df_filmes = (df_filmes.withColumn("rn", F.row_number().over(janela_filmes)).filter(F.col("rn") == 1).drop("rn"))

# Converte a coluna original 'release_date' testando os diferentes padrões
df_filmes = df_filmes.withColumn(
    "release_date_tratada",
    F.coalesce(
        F.try_to_date(F.col("release_date"), "yyyy-MM-dd"),
        F.try_to_date(F.col("release_date"), "MM-dd-yyyy"),
        F.try_to_date(F.col("release_date"), "dd/MM/yyyy"),
        F.try_to_date(F.col("release_date"), "yyyy/MM/dd"),
        F.try_to_date(F.col("release_date"), "dd-MM-yyyy"),
        F.try_to_date(F.col("release_date"), "MM/dd/yyyy")
    )
)

# Criar a coluna 'ano_lancamento', extraída da data tratada
df_filmes = df_filmes.withColumn("ano_lancamento", F.year(F.col("release_date_tratada")))

# Mapeamento de Colunas, renomeação para Português e tipagem

df_silver = df_filmes.select(
    F.expr("try_cast(id AS INT)").alias("id_filme"),
    F.col("title").alias("titulo"),
    F.col("original_title").alias("titulo_original"),
    F.col("release_date_tratada").alias("data_lancamento"),
    F.col("ano_lancamento"),
    F.expr("try_cast(runtime AS INT)").alias("duracao_minutos"),

    # Mantém apenas códigos de idioma com duas letras
    F.when(
        F.col("original_language").rlike(r"^[A-Za-z]{2}$"),
        F.lower(F.col("original_language"))
    ).otherwise(None).alias("idioma_original"),

    F.col("status_filme"),
    F.col("overview").alias("sinopse"),
    F.col("tagline").alias("frase_divulgacao")
)

# Conferência visual dos dados finais da camada Silver
# display(df_silver.limit(20))

# Gravação na Camada Silver
df_silver.write.format("delta").mode("overwrite").saveAsTable("silver.tb_info_filmes")

# Confere a tabela após a gravação
display(spark.table("silver.tb_info_filmes").limit(20))

id_filme,titulo,titulo_original,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse,frase_divulgacao
1000004,Purple Beatz,Purple Beatz,2022-07-07,2022,86,en,Lançado,"Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry.",A drum n bass romance.
1000005,Aisha Brown: The First Black Woman Ever,Aisha Brown: The First Black Woman Ever,2020-02-14,2020,42,en,Lançado,"No one, and nothing, is off-limits for comedian Aisha Brown, as she takes on her boyfriend’s penis, racism, clinical depression, and Donald Trump, in this hilarious one-hour comedy special from Just For Laughs.",null
1000007,KYLE BROWNRIGG: INTRODUCING LYLE,Kyle Brownrigg: Introducing Lyle,2022-05-27,2022,36,en,Lançado,"Kyle Brownrigg takes the stage in this hilarious half-hour stand-up special where he laments gender reveal parties, talks about his Irish boyfriend, and introduces the world to his drunk persona, Lyle.",null
1000011,Worth Your Weight in Gold,O Teu Peso Em Ouro,2022-07-14,2022,26,pt,Lançado,"Oscar, a renowned hypnotherapist, uses the last moments in his hotel room to say goodbye to the vertiginous Mercedes and rid old Norberto of his bitterness.",null
1000014,On va manquer !,On va manquer !,2018-05-15,2018,0,fr,Lançado,null,null
1000030,58 Hours: The Baby Jessica Story,58 Hours: The Baby Jessica Story,2021-07-31,2021,0,es,Lançado,null,null
1000054,One Hundred Years and Hope,百年と希望,2022-06-18,2022,107,ja,Lançado,"In a country ruled by the Liberal Democratic Party, running on austerity and neoliberal ambitions, for most of its postwar years, gender and economic inequalities have become increasingly acute in Japan. Takashi Nishihara, a filmmaker who has been following the youth protests in Japan notices that there is one party that seems to be raising issues of gender and economic in the political sphere, the Japanese Communist Party (JCP), a party about to enter its hundredth year and consistently burdened by its historical connotations. Though an outsider of the party, Nishihara gained unprecedented access to the JCP and driven by his interest in the younger party members who find hope in the JCP, the resulting documentary goes beyond party politics and observes the current grassroots leftist movements in Japan. It also becomes witness to the larger and deep-seated patriarchal system that continues to quell momentums of hope.",null
1000058,Homecoming,Le retour,2023-07-12,2023,110,fr,Lançado,"Kheìdidja, in her forties, works for a wealthy Parisian family who offers her the opportunity to take care of their children for a summer in Corsica. It's an opportunity for her to return with her daughters, Jessica and Farah, to the island they left fifteen years earlier in tragic circumstances.",null
1000059,素敵な選TAXI SPECIAL〜湯けむり連続選択肢〜,素敵な選TAXI SPECIAL〜湯けむり連続選択肢〜,2016-04-05,2016,116,ja,Lançado,"Edawakare, the driver of the Time Taxi that allows passengers to return to their life's turning points, visits a hot spring this time around. An array of guests at the inn are fraught with troubles in their life and become passengers of the Time Taxi. And somehow all the clients are actually part of a bigger picture?!",null
1000073,A Chance To Win,Pour l'honneur,2023-05-03,2023,97,fr,Lançado,"Two villages in the south of France have always been bitter rivals, but when a group of asylum seekers arrive in the community, the life of both villages is shaken up and age-old disagreements escalate. Their antagonism reaches its peak with the annual rugby derby played between the two village teams, but this time, with the new outsiders joining as unexpected recruits, the result of the 100th match will be more unpredictable than ever.",null


## **Camada Silver - silver.tb_financeiro_filmes (origem: tb_movies_financials)**

In [0]:
spark.sql("DROP TABLE IF EXISTS silver.tb_financeiro_filmes")

# Lê os dados financeiros da camada Bronze
df_financeiro = spark.table("bronze.tb_movies_financials")

# Exibe o schema para identificarmos os tipos das colunas
# df_financeiro.printSchema()

# Exibe algumas linhas para verificarmos o formato dos valores monetários
display(df_financeiro.limit(20))

id,budget,revenue,ingestion_datetime
293660,58000000,Unknown,2026-10-07T07:55:37.850Z
299536,300000000,2052415039,2026-10-07T07:55:37.850Z
299534,356000000,2800000000,2026-10-07T07:55:37.850Z
475557,55000000,1074458282,2026-10-07T07:55:37.850Z
271110,250000000,Não Informado,2026-10-07T07:55:37.850Z
284054,200000000,1349926083,2026-10-07T07:55:37.850Z
284052,180000000,676343174,2026-10-07T07:55:37.850Z
315635,175000000,880166924,2026-10-07T07:55:37.850Z
283995,200000000,863756051,2026-10-07T07:55:37.850Z
297761,175000000,746846894,2026-10-07T07:55:37.850Z


In [0]:
# Lê a cotação do dólar da camada Bronze
# df_cotacao = spark.table("bronze.tb_cotacao_dolar")

# Exibe o schema e algumas linhas para conferirmos os dados
# df_cotacao.printSchema()
# display(df_cotacao.limit(20))

df_cotacao = spark.table("bronze.tb_cotacao_dolar")

taxa_dolar = (df_cotacao
    .orderBy(F.col("dataHoraCotacao").desc())
    .select("cotacaoVenda")
    .first()[0]
)

In [0]:
# Renomeia as colunas 
df_financeiro = (df_financeiro
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("budget", "orcamento_raw")
    .withColumnRenamed("revenue", "receita_raw")
)

# Valores que representam dados ausentes
valores_ausentes = ["unknown", "não informado", "n/a", "na", "none", ""]

# Limpa os valores financeiros e converte para número
for coluna in ["orcamento_raw", "receita_raw"]:df_financeiro = df_financeiro.withColumn(coluna,
        F.when(
            F.trim(F.lower(F.col(coluna))).isin(valores_ausentes),
            None
        ).otherwise(
            F.regexp_replace(F.col(coluna), r"[R$\s,]", "")
        )
    )

# Converte os valores para decimal
df_financeiro = (df_financeiro
    .withColumn("orcamento_usd", F.expr("try_cast(orcamento_raw AS DECIMAL(18,2))"))
    .withColumn("receita_usd", F.expr("try_cast(receita_raw AS DECIMAL(18,2))"))
)

# Valores zero ou negativos são considerados inválidos
df_financeiro = (df_financeiro
    .withColumn(
        "orcamento_usd",
        F.when(F.col("orcamento_usd") > 0, F.col("orcamento_usd"))
    )
    .withColumn(
        "receita_usd",
        F.when(F.col("receita_usd") > 0, F.col("receita_usd"))
    )
)

# Converte USD para BRL
df_financeiro = (df_financeiro
    .withColumn("orcamento_brl", F.round(F.col("orcamento_usd") * taxa_dolar, 2))
    .withColumn("receita_brl", F.round(F.col("receita_usd") * taxa_dolar, 2))
)

# Calcula lucro e margem de lucro
df_financeiro = (df_financeiro
    .withColumn(
        "lucro_usd",
        F.when(
            F.col("receita_usd").isNotNull() & F.col("orcamento_usd").isNotNull(),
            F.round(F.col("receita_usd") - F.col("orcamento_usd"), 2)
        )
    )
    .withColumn(
        "lucro_brl",
        F.when(
            F.col("receita_brl").isNotNull() & F.col("orcamento_brl").isNotNull(),
            F.round(F.col("receita_brl") - F.col("orcamento_brl"), 2)
        )
    )
    .withColumn(
        "margem_lucro_pct",
        F.when(
            F.col("orcamento_usd") > 0,
            F.round((F.col("lucro_usd") / F.col("orcamento_usd")) * 100, 2)
        )
    )
)

# Seleciona somente as colunas finais
df_financeiro = df_financeiro.select(
    "id_filme",
    "orcamento_usd",
    "receita_usd",
    "orcamento_brl",
    "receita_brl",
    "lucro_usd",
    "lucro_brl",
    "margem_lucro_pct"
)

# Salva a tabela financeira na camada Silver
df_financeiro.write.format("delta").mode("overwrite").saveAsTable("silver.tb_financeiro_filmes")

display(spark.table("silver.tb_financeiro_filmes").limit(20))

id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_pct
293660,58000000.00,null,2.882484E8,null,null,null,null
299536,300000000.00,2052415039.00,1.49094E9,1.020009226082E10,1752415039.00,8.70915226082E9,584.14
299534,356000000.00,2800000000.00,1.7692488E9,1.391544E10,2444000000.00,1.21461912E10,686.52
475557,55000000.00,1074458282.00,2.73339E8,5.33984276988E9,1019458282.00,5.06650376988E9,1853.56
271110,250000000.00,null,1.24245E9,null,null,null,null
284054,200000000.00,1349926083.00,9.9396E8,6.70886264729E9,1149926083.00,5.71490264729E9,574.96
284052,180000000.00,676343174.00,8.94564E8,3.36129030615E9,496343174.00,2.46672630615E9,275.75
315635,175000000.00,880166924.00,8.69715E8,4.3742535789E9,705166924.00,3.5045385789E9,402.95
283995,200000000.00,863756051.00,9.9396E8,4.29269482226E9,663756051.00,3.29873482226E9,331.88
297761,175000000.00,746846894.00,8.69715E8,3.7116796938E9,571846894.00,2.8419646938E9,326.77


## **Camada Silver -  silver.tb_metricas_engajamento (origem: tb_movies_metrics)**

In [0]:
# Lê os dados de métricas da camada Bronze
df_metricas = spark.table("bronze.tb_movies_metrics")

# Exibe algumas linhas para verificarmos o formato dos valores monetários
display(df_metricas.limit(30))


df_metricas.printSchema()

id,popularity,vote_average,vote_count,averageRating,numVotes,ingestion_datetime
293660,72.735,7.606,28894,8.0,1270339,2026-10-07T07:55:42.173Z
299536,"154,34",8.255,27713,8.4,1406782,2026-10-07T07:55:42.173Z
299534,91.756,8.263,23857,8.4,1484150,2026-10-07T07:55:42.173Z
475557,"54,522",8.168,23425,8.3,1723035,2026-10-07T07:55:42.173Z
271110,70.741,7.4,21541,7.8,947222,2026-10-07T07:55:42.173Z
284054,43.665,7.39,null,7.3,924922,2026-10-07T07:55:42.173Z
284052,70.535,7.427,20935,7.5,895880,2026-10-07T07:55:42.173Z
315635,65.88,7.345,20507,7.4,835116,2026-10-07T07:55:42.173Z
283995,67.553,7.624,20353,7.6,844767,2026-10-07T07:55:42.173Z
297761,35.356,5.909,20097,5.9,null,2026-10-07T07:55:42.173Z


root
 |-- id: string (nullable = true)
 |-- popularity: string (nullable = true)
 |-- vote_average: string (nullable = true)
 |-- vote_count: string (nullable = true)
 |-- averageRating: string (nullable = true)
 |-- numVotes: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



In [0]:
spark.sql("DROP TABLE IF EXISTS silver.tb_metricas_engajamento")

df_met = spark.table("bronze.tb_movies_metrics")

# Renomeia as colunas
df_met = (
    df_met
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("popularity", "popularidade_raw")
    .withColumnRenamed("vote_average", "nota_media_tmdb_raw")
    .withColumnRenamed("vote_count", "qtd_votos_tmdb_raw")
    .withColumnRenamed("averageRating", "nota_media_imdb_raw")
    .withColumnRenamed("numVotes", "qtd_votos_imdb_raw")
)

# Limpa a popularidade antes da conversão de tipo
# Normaliza a vírgula como separador decimal
df_met = df_met.withColumn("popularidade",
    F.expr("""try_cast(replace(trim(popularidade_raw), ',', '.') AS DOUBLE)""")
)

# Converte as métricas para tipos numéricos
df_met = (df_met
    .withColumn(
        "nota_media_tmdb",
        F.expr("try_cast(nota_media_tmdb_raw AS DOUBLE)")
    )
    .withColumn(
        "qtd_votos_tmdb",
        F.expr("try_cast(qtd_votos_tmdb_raw AS BIGINT)")
    )
    .withColumn(
        "nota_media_imdb",
        F.expr("try_cast(nota_media_imdb_raw AS DOUBLE)")
    )
    .withColumn(
        "qtd_votos_imdb",
        F.expr("try_cast(qtd_votos_imdb_raw AS BIGINT)")
    )
)

# Aplica os limites de negócio
df_met = (df_met
    .withColumn(
        "nota_media_tmdb",
        F.when(F.col("nota_media_tmdb").between(0, 10),
               F.col("nota_media_tmdb"))
    )
    .withColumn(
        "nota_media_imdb",
        F.when(F.col("nota_media_imdb").between(0, 10),
               F.col("nota_media_imdb"))
    )
    .withColumn(
        "qtd_votos_tmdb",
        F.when(F.col("qtd_votos_tmdb") >= 0,
               F.col("qtd_votos_tmdb"))
    )
    .withColumn(
        "qtd_votos_imdb",
        F.when(F.col("qtd_votos_imdb") >= 0,
               F.col("qtd_votos_imdb"))
    )
    .withColumn(
        "popularidade",
        F.when(F.col("popularidade") >= 0,
               F.col("popularidade"))
    )
)

# Seleciona as colunas finais
df_met_final = df_met.select(
    "id_filme",
    "popularidade",
    "nota_media_tmdb",
    "qtd_votos_tmdb",
    "nota_media_imdb",
    "qtd_votos_imdb"
)

# Salva na Silver
df_met_final.write.format("delta").mode("overwrite").saveAsTable("silver.tb_metricas_engajamento")

# Display
display(spark.table("silver.tb_metricas_engajamento").limit(10))


id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
293660,72.735,7.606,28894,8.0,1270339
299536,154.34,8.255,27713,8.4,1406782
299534,91.756,8.263,23857,8.4,1484150
475557,54.522,8.168,23425,8.3,1723035
271110,70.741,7.4,21541,7.8,947222
284054,43.665,7.39,null,7.3,924922
284052,70.535,7.427,20935,7.5,895880
315635,65.88,7.345,20507,7.4,835116
283995,67.553,7.624,20353,7.6,844767
297761,35.356,5.909,20097,5.9,null


## **Camada Silver -  silver.tb_avaliacoes_usuarios (origem: tb_movies_reviews)**

In [0]:
# Remove a tabela anterior, caso ela já exista
spark.sql("DROP TABLE IF EXISTS silver.tb_avaliacoes_usuarios")

# Carrega a tabela de avaliações da camada Bronze
df_avaliacoes = spark.table("bronze.tb_movies_reviews")

#  Visualiza alguns registros para identificar o formato dos dados
display(df_avaliacoes.limit(30))


df_avaliacoes.printSchema()

id,nome,nota,comentario,ingestion_datetime
442113,Mariana Cardoso 277,4.4,null,2026-10-07T07:55:51.274Z
637007,Lucas Reis 602,3.9,null,2026-10-07T07:55:51.274Z
449479,Sérgio Freitas,0.7,Péssimo em todos os sentidos.,2026-10-07T07:55:51.274Z
413036,Gabriela Monteiro 401,7.5,null,2026-10-07T07:55:51.274Z
528480,Leonardo Monteiro,6.3,Assisti até o final mas não me marcou.,2026-10-07T07:55:51.274Z
387727,Maria Alves 707,8.2,"Gostei bastante, recomendo.",2026-10-07T07:55:51.274Z
1032506,Amanda Castro 242,4.7,"Fraco, não recomendo.",2026-10-07T07:55:51.274Z
446554,Sandra Rodrigues 736,6.5,Assisti até o final mas não me marcou.,2026-10-07T07:55:51.274Z
569916,Camila Lima 489,8.2,Muito bom! Vale a pena assistir.,2026-10-07T07:55:51.274Z
864552,Roberto Almeida 555,9.3,"Obra-prima do cinema, simplesmente espetacular.",2026-10-07T07:55:51.274Z


root
 |-- id: string (nullable = true)
 |-- nome: string (nullable = true)
 |-- nota: string (nullable = true)
 |-- comentario: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



In [0]:
# Renomeia as colunas conforme o modelo da camada Silver
df_avaliacoes = (
    df_avaliacoes
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("nome", "nome_usuario")
    .withColumnRenamed("nota", "nota_usuario")
    .withColumnRenamed("comentario", "comentario_usuario")
)

# Converte a nota para número
df_avaliacoes = df_avaliacoes.withColumn(
    "nota_usuario",
    F.expr("try_cast(nota_usuario AS DOUBLE)")
)

# Validar nota: deve estar entre 0 e 10
df_avaliacoes = df_avaliacoes.withColumn(
    "nota_usuario",
    F.when(
        (F.col("nota_usuario") >= 0) & (F.col("nota_usuario") <= 10),F.col("nota_usuario"))
    .otherwise(F.lit(None))
)

# Substitui comentários nulos ou compostos apenas por espaços pelo texto padronizado "Sem comentário"
df_avaliacoes = df_avaliacoes.withColumn(
    "comentario_usuario",
    F.when(
        F.col("comentario_usuario").isNull() |
        (F.trim(F.col("comentario_usuario")) == ""),"Sem comentário")
    .otherwise(F.col("comentario_usuario"))
)

# Remover duplicatas integrais
df_avaliacoes = df_avaliacoes.dropDuplicates(["id_filme", "nome_usuario", "nota_usuario", "comentario_usuario"])

# Seleciona somente as colunas necessárias na camada Silver
df_avaliacoes_final = df_avaliacoes.select(
    "id_filme",
    "nome_usuario",
    "nota_usuario",
    "comentario_usuario"
)

# Salva a tabela de avaliações na camada Silver
df_avaliacoes_final.write.format("delta").mode("overwrite").saveAsTable("silver.tb_avaliacoes_usuarios")

# Exibe alguns registros da tabela criada
display(spark.table("silver.tb_avaliacoes_usuarios").limit(20))

id_filme,nome_usuario,nota_usuario,comentario_usuario
637007,Lucas Reis 602,3.9,Sem comentário
1100094,Gabriel Carvalho 581,6.2,"Aceitável, mas esperava mais."
628575,Alexandre Barbosa 220,0.3,Péssimo em todos os sentidos.
573249,Rodrigo Oliveira 273,0.5,Péssimo em todos os sentidos.
592539,Pedro Costa 181,4.8,"Não gostei, história confusa."
464493,Adriana Dias 257,0.4,Péssimo em todos os sentidos.
1199748,Eduardo Dias 177,6.0,"Poderia ser melhor, mas não é ruim."
640543,Cristina Monteiro 310,6.4,Sem comentário
599134,Larissa Lopes 330,2.6,Péssimo em todos os sentidos.
424011,Vinícius Ferreira 405,0.5,Não recomendo de jeito nenhum.


## **Camada Silver -  silver.tb_generos (origem: tb_credits_and_tags, coluna genres)**

In [0]:
spark.sql("DROP TABLE IF EXISTS silver.tb_generos")

# Carrega a tabela de generos e tags da camada Bronze
df_creditos = spark.table("bronze.tb_credits_and_tags")

# Visualiza alguns registros para identificar os formatos presentes na coluna genres
display(
    df_creditos.select("id", "genres").limit(30)
)

# Verifica os tipos das colunas da tabela
df_creditos.printSchema()

id,genres
293660,"Action, Adventure, Comedy"
299536,"Adventure, Action, Science Fiction"
299534,"Adventure, Science Fiction, Action"
475557,"Crime, Thriller, Drama"
271110,"Adventure, Action, Science Fiction"
284054,"Action, Adventure, Science Fiction"
284052,"Action, Adventure, Fantasy"
315635,"Action, Adventure, Science Fiction, Drama"
283995,"Science Fiction, Adventure, Action"
297761,Action|Adventure|Fantasy


root
 |-- id: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- production_companies: string (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- spoken_languages: string (nullable = true)
 |-- keywords: string (nullable = true)
 |-- directors: string (nullable = true)
 |-- writers: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



In [0]:
# Normaliza os separadores da coluna genres
df_genres = df_creditos.select(
    F.col("id").alias("id_filme"),
    F.regexp_replace(F.col("genres"), r"[;|]", ",").alias("genres_normalized")
)

# Divide os gêneros e cria um registro para cada gênero
df_genres = (
    df_genres
    .withColumn("genero_array", F.split(F.col("genres_normalized"), ","))
    .withColumn("nome_genero", F.explode(F.col("genero_array")))
    .withColumn("nome_genero", F.trim(F.col("nome_genero")))
)

# Padroniza a capitalização antes de comparar com a lista
df_genres = df_genres.withColumn("nome_genero", F.initcap(F.col("nome_genero")))

# Domínio oficial de gêneros (19 gêneros do TMDB)
generos_validos = [
    "Action", "Adventure", "Animation", "Comedy", "Crime",
    "Documentary", "Drama", "Family", "Fantasy", "History",
    "Horror", "Music", "Mystery", "Romance", "Science Fiction",
    "Tv Movie", "Thriller", "War", "Western"
]

# Filtra estritamente o domínio de negócio
df_genres = df_genres.filter(F.col("nome_genero").isin(generos_validos))

# Seleciona apenas as colunas oficiais da Silver
df_generos_final = df_genres.select("id_filme","nome_genero")

# Remove duplicatas
df_generos_final = df_generos_final.dropDuplicates(["id_filme", "nome_genero"])

# Salva na camada Silver
df_generos_final.write.format("delta").mode("overwrite").saveAsTable("silver.tb_generos")

display(spark.table("silver.tb_generos").limit(20))


id_filme,nome_genero
329865,Mystery
269149,Animation
429617,Science Fiction
330459,Action
127380,Family
332562,Romance
291805,Action
293167,Action
381284,History
454626,Action


## **Camada Silver -  silver.tb_pessoas_empresas (origem: tb_credits_and_tags, colunas cast, directors, writers,production_companies)**

In [0]:
spark.sql("DROP TABLE IF EXISTS silver.tb_pessoas_empresas")

# Carrega a tabela de pessoas e tags da camada Bronze
df_credits = spark.table("bronze.tb_credits_and_tags")



In [0]:
# Mapeia coluna de origem para tipo de entidade
mapping = {
    "cast": "Ator",
    "directors": "Diretor",
    "writers": "Roteirista",
    "production_companies": "Produtora"
}

dfs = []

for coluna, tipo in mapping.items():

    df = (df_creditos
        .select(
            F.col("id").alias("id_filme"),
            F.col(coluna).alias("nome_entidade")
        )
        .withColumn("nome_entidade", F.explode(F.split("nome_entidade", ",")))
        .withColumn("nome_entidade", F.initcap(F.trim(F.col("nome_entidade"))))
        .withColumn("tipo_entidade", F.lit(tipo))
        .filter(
            F.col("nome_entidade").isNotNull() & (F.col("nome_entidade") != "") &
            (~F.lower(F.col("nome_entidade")).isin("n/a", "na", "none", "unknown")) &
            (~F.col("nome_entidade").rlike(r"^[0-9.]+$"))
        )
    )

    dfs.append(df)

# Junta os quatro tipos de entidade
df_pessoas = dfs[0]
for df in dfs[1:]:
    df_pessoas = df_pessoas.unionByName(df)

# Remove duplicatas
df_pessoas = df_pessoas.dropDuplicates(
    ["id_filme", "nome_entidade", "tipo_entidade"]
)

# Salva a tabela Silver
df_pessoas.write.format("delta").mode("overwrite").saveAsTable(
    "silver.tb_pessoas_empresas"
)

display(spark.table("silver.tb_pessoas_empresas").limit(20))

id_filme,nome_entidade,tipo_entidade
299536,Don Cheadle,Ator
263115,Hugh Jackman,Ator
354912,Anthony Gonzalez,Ator
419430,Daniel Kaluuya,Ator
424694,Mike Myers,Ator
374720,Mark Rylance,Ator
321612,Ewan Mcgregor,Ator
335983,Tom Hardy,Ator
363088,Evangeline Lilly,Ator
363088,Randall Park,Ator


## **Camada Silver -  silver.tb_cotacao_dolar (origem: tb_cotacao_dolar)**

In [0]:
# Remove a tabela anterior, caso exista
spark.sql("DROP TABLE IF EXISTS silver.tb_cotacao_dolar")

# Lê a tabela Bronze
df_cotacao = spark.table("bronze.tb_cotacao_dolar")

# display(spark.table("bronze.tb_cotacao_dolar").orderBy("dataHoraCotacao"))

In [0]:
# Converte a data/hora para apenas a data
df_cotacao = df_cotacao.withColumn("data_cotacao",F.to_date(F.col("dataHoraCotacao")))

# Cria o intervalo completo de datas
datas = df_cotacao.select(
    F.min("data_cotacao").alias("data_inicial"),
    F.max("data_cotacao").alias("data_final")
).select(
    F.explode(
        F.sequence(
            F.col("data_inicial"),
            F.col("data_final"),
            F.expr("interval 1 day")
        )
    ).alias("data_cotacao")
)

# Junta todas as datas com as cotações disponíveis
df_cotacao = datas.join(df_cotacao.select(
        "data_cotacao",
        "cotacaoCompra",
        "cotacaoVenda"
    ).dropDuplicates(["data_cotacao"]),
    on="data_cotacao",
    how="left"
)

# Usa a última cotação disponível para preencher datas sem cotação
window = (Window
    .orderBy("data_cotacao")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_cotacao = (df_cotacao
    .withColumn(
        "cotacao_compra",
        F.last("cotacaoCompra", ignorenulls=True).over(window)
    )
    .withColumn(
        "cotacao_venda",
        F.last("cotacaoVenda", ignorenulls=True).over(window)
    )
)

# Seleciona apenas as colunas finais da Silver
df_cotacao_final = df_cotacao.select(
    "data_cotacao",
    "cotacao_compra",
    "cotacao_venda"
)

# Salva a tabela Silver
df_cotacao_final.write.format("delta").mode("overwrite").saveAsTable("silver.tb_cotacao_dolar")

# Display
display(spark.table("silver.tb_cotacao_dolar").orderBy("data_cotacao"))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


data_cotacao,cotacao_compra,cotacao_venda
2026-09-30,5.1803,5.1809
2026-10-01,5.2073,5.2079
2026-10-02,5.2232,5.2238
2026-10-03,5.2232,5.2238
2026-10-04,5.2232,5.2238
2026-10-05,4.9853,4.9859
2026-10-06,4.9692,4.9698
